In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

### Scenario:
You're given `shipment_status_events.json` — a long, narrow **event log** where each shipment generates one row per status change (`CREATED`, `PICKED_UP`, `IN_TRANSIT`, `DELIVERED`). Operations wants a **wide "current state" view**: one row per shipment, with each status as its own column showing when that status was reached. Some shipments haven't reached every status yet (still in transit), and one shipment skipped a stage entirely (no `PICKED_UP` event) — both cases should just show as `null` in that status's column, not break the transformation.

**Problem:**

- Read the JSON file with an explicit schema.
- **Pivot** the data: one row per `shipment_id`, with a separate column for each distinct `status` value, containing that status's `status_time`. Use `.groupBy("shipment_id").pivot("status")` for this — don't try to do it with manual joins or conditional aggregation.
- Add a `transit_hours` column: the difference between `DELIVERED` and `CREATED` timestamps, in hours. If a shipment hasn't been delivered yet, this should be `null`, not an error.
- Order the output by `shipment_id` ascending.

**Schema**

| Column | Type |
| :--- | :--- |
| **shipment_id** | string |
| **status** | string |
| **status_time** | timestamp |

**Expected Output**

| shipment_id | CREATED | DELIVERED | IN_TRANSIT | PICKED_UP | transit_hours |
| :--- | :--- | :--- | :--- | :--- | :--- |
| SH1001 | 2025-02-01 08:00:00 | 2025-02-02 09:00:00 | 2025-02-01 14:00:00 | 2025-02-01 10:00:00 | 25.0 |
| SH1002 | 2025-02-01 09:00:00 | null | 2025-02-01 15:00:00 | 2025-02-01 11:30:00 | null |
| SH1003 | 2025-02-01 07:30:00 | 2025-02-01 20:00:00 | 2025-02-01 12:00:00 | null | 12.5 |

In [0]:
source_df = spark.read.json("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/shipment_status_events.json")

current_sattus_df = source_df.groupBy(col("shipment_id")).pivot("status").agg(max(col("status_time")))
current_sattus_df = current_sattus_df.withColumn(
    "transit_hours", 
    when(col("DELIVERED").isNotNull(), (unix_timestamp(col("DELIVERED")) - unix_timestamp(col("CREATED")))/3600)
).orderBy(col("shipment_id"))
current_sattus_df.show()

### Scenario:
You're given `employee_skills_pipe.txt` — a **mixed-delimiter** text file. Fields are separated by `|`, but one of those fields (`skills`) is itself a **comma-separated sub-list** embedded inside a single pipe-delimited field. This layered-delimiter shape shows up constantly in legacy exports and EDI-style files — you can't parse it with a single `split()` call; you need to split on the outer delimiter first, then split again on the inner one for just that field.

**Problem:**

- Read the file as plain text, treating the first line as a header you'll need to skip (don't rely on `spark.read.csv`'s header handling here — parse it manually with `split()` since the delimiter isn't a comma).
- Split each line on `|` to get `emp_id`, `name`, `department`, and `skills` (the raw comma-separated string).
- Split the `skills` field on `,` to turn it into an array, then **explode** it so each employee-skill pair becomes its own row.
- Count how many distinct employees have each skill.
- Order by count descending, then skill ascending.

**Expected Output**

| skill | employee_count |
| :--- | :--- |
| Python | 3 |
| SQL | 3 |
| Java | 2 |
| Spark | 2 |
| Tableau | 1 |

In [0]:
emp_skills_df = spark.read.text("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/employee_skills_pipe.txt")

header = emp_skills_df.first()["value"]
columns = header.split("|")

data_df = emp_skills_df.filter(col("value")!= header).withColumn("data_parts", split(col("value"), r"\|"))
emp_skills_df = data_df.select(
    col("data_parts")[0].alias(columns[0]),
    col("data_parts")[1].alias(columns[1]),
    col("data_parts")[2].alias(columns[2]),
    col("data_parts")[3].alias(columns[3])
)

emp_skills_df = emp_skills_df.withColumn("skills", split(col("skills"), ","))
emp_skills_df = emp_skills_df.select("emp_id", "name", "department", explode("skills").alias("skill"))
employee_count_df = emp_skills_df.groupBy("skill").agg(countDistinct("emp_id").alias("employee_count"))
employee_count_df.orderBy(col("employee_count").desc(), col("skill").asc()).show()

### Scenario:
You're given `product_attributes.json` — each product has an `attributes` object, but the **keys inside it are completely different per product** (a t-shirt has `color`/`size`, a heater has `voltage`/`wattage`, a mug has `color`/`material`). If you let Spark infer this normally, it treats `attributes` as a **struct** and unions every key it's ever seen across every record into one fixed schema — meaning as more product types get added over time, that struct schema keeps growing forever, and most rows end up mostly `null`. The correct real-world fix is to read a field like this as a **`MapType`** instead of letting it become a struct — a map handles arbitrary, varying keys natively without any schema explosion.

**Problem — do both parts:**

**Part 1 — Read as a map, then flatten to long format:**
- Define an explicit schema where `attributes` is typed as `MapType(StringType(), StringType())` — don't let Spark infer this field.
- Read the file with that schema.
- Explode the `attributes` map so each product-attribute pair becomes its own row, with the map key and value as separate columns.
- Final columns: `product_id`, `product_name`, `attr_key`, `attr_value`.
- Order by `product_id` ascending, then `attr_key` ascending.

**Part 2 — Count products with a specific attribute:**
- From the same exploded data, count how many **distinct products** have a `color` attribute defined at all (regardless of its value).

**Schema**

| Column | Type |
| :--- | :--- |
| **product_id** | string |
| **product_name** | string |
| **attributes** | map\<string, string\> |

**Expected Output — Part 1**

| product_id | product_name | attr_key | attr_value |
| :--- | :--- | :--- | :--- |
| P001 | T-Shirt | color | Red |
| P001 | T-Shirt | size | M |
| P002 | Heater | voltage | 220V |
| P002 | Heater | wattage | 1500W |
| P003 | Mug | color | Blue |
| P003 | Mug | material | Ceramic |

**Expected Output — Part 2**

| products_with_color |
| :--- |
| 2 |

In [0]:
schema = StructType(
    [
        StructField("product_id", StringType()),
        StructField("product_name", StringType()),
        StructField("attributes", MapType(StringType(), StringType()))
    ]
)

product_attr_json = spark.read.schema(schema).json("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/product_attributes.json")

product_attr_df = product_attr_json.select("product_id", "product_name", explode("attributes").alias("attr_key", "attr_value"))
product_attr_df.orderBy(col("product_id").asc(), col("attr_key").asc()).show()
color_attr_count = product_attr_df.filter(col("attr_key") == "color").count()
print(f"products_with_color: {color_attr_count}")